# Detector ternário de mosquitos — guia didático

Este caderno conduz o fluxo de ponta a ponta: **entender os dados → conferir o protocolo → treinar → testar em grupos separados → aplicar em um WAV**. As explicações e a execução estão reunidas aqui. Módulos Python pequenos permanecem como implementação auditável das funções reutilizadas pelo treino e pela inferência.

As três saídas são `no_mosquito`, `other_mosquito` e `aedes_aegypti`. Este é um experimento offline. Os números abaixo descrevem o corpus avaliado e não demonstram confiabilidade em novas regiões, microfones ou condições de campo.

**Como acompanhar:** execute as células em ordem. Por padrão, o caderno mostra o resultado de referência sem treinar. Para iniciar um treino novo, altere `RUN_NEW_TRAINING` para `True` na célula de configuração. Uma rodada completa pode demorar bastante e cria uma pasta nova em `results/`.

## 1. O que este experimento aprende

Cada linha da tabela corresponde a uma janela de áudio já transformada em descritores acústicos. O modelo combina regressão logística e HistGradientBoosting. O problema tem três classes: silêncio/ruído sem mosquito, outros mosquitos e *Aedes aegypti*.

A pergunta não é apenas “quantas janelas acertou?”. Janelas do mesmo arquivo se parecem. Por isso, a unidade principal do teste é o **grupo de gravação**, e blocos identificados como o mesmo áudio PCM são mantidos na mesma partição. Isso reduz vazamento entre treino e teste.

O pipeline usa validação cruzada aninhada: as dobras internas escolhem configuração e regras de decisão; a dobra externa estima desempenho em grupos que não participaram dessas escolhas. Depois do teste, um modelo final é ajustado em todos os dados de desenvolvimento para permitir a aplicação a uma gravação. Esse modelo final não tem um teste independente próprio.

## 2. Preparar o ambiente e localizar o projeto

Na raiz do repositório, instale as dependências listadas em `requirements.txt`. Os arquivos de áudio e o cache de descritores não são copiados para o GitHub. O treino completo exige que eles estejam disponíveis localmente nos caminhos exibidos na próxima célula.

Os metadados compactos das janelas e os vínculos entre gravações acompanham o repositório em `data/metadata/`. O cache `features.npy` deve ter uma linha por janela do manifesto e ser preparado a partir dos áudios originais com o frontend descrito neste projeto.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay

WORKING_DIR = Path.cwd().resolve()
ROOT = next((p for p in [WORKING_DIR, *WORKING_DIR.parents]
             if (p / "train_ternary_nested_models.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Abra este caderno dentro de uma cópia do projeto mosquito-wingbeat.")
sys.path.insert(0, str(ROOT))

from ternary_protocol import ROOT as PROJECT_ROOT, SEGMENTS, FEATURES, KNOWN_MAP, CLASS_NAMES
assert ROOT == PROJECT_ROOT

# Controles da execução: não sobrescrevemos rodadas anteriores.
RUN_NEW_TRAINING = False
THREADS = 4
AUDIO_PATH = None  # Ex.: Path("/caminho/para/gravacao.wav")

print("Raiz do projeto:", ROOT)
print("Manifesto de janelas:", SEGMENTS)
print("Cache de descritores:", FEATURES)
print("Mapa de colisões PCM:", KNOWN_MAP)

## 3. Conferir as entradas antes do treino

O manifesto contém rótulos, grupo de gravação, domínio, posição temporal e índice da janela. O mapa de colisões reúne gravações que compartilham a mesma identidade PCM conhecida. O cache é uma matriz numérica alinhada linha a linha ao manifesto.

**Por que o cache não está no Git?** Ele é um artefato reconstruível e os áudios originais são volumosos. Para manter o repositório leve, o notebook interrompe o treino com uma mensagem clara se o cache local estiver ausente; os metadados necessários para interpretar/alocar as linhas continuam versionados.

In [ ]:
required_inputs = {
    "manifesto de janelas": SEGMENTS,
    "cache de descritores": FEATURES,
    "mapa de colisões PCM": KNOWN_MAP,
}
missing = [f"{name}: {path}" for name, path in required_inputs.items() if not path.is_file()]
if RUN_NEW_TRAINING and missing:
    raise FileNotFoundError("Para treinar, providencie estas entradas locais:\n- " + "\n- ".join(missing))

if SEGMENTS.is_file():
    manifest = pd.read_csv(SEGMENTS, low_memory=False)
    print(f"Janelas no manifesto: {len(manifest):,}")
    display(manifest[[c for c in ["path", "label", "group", "domain", "start_s", "bucket"] if c in manifest]].head())
if FEATURES.is_file():
    feature_shape = np.load(FEATURES, mmap_mode="r").shape
    print("Formato do cache:", feature_shape)
if KNOWN_MAP.is_file():
    print("Mapa de colisões disponível:", KNOWN_MAP.stat().st_size, "bytes")

## 4. Um resultado de referência — sem retreino

Este retrato foi calculado a partir da avaliação OOF concluída em 14/10/2026. O caderno guarda os números essenciais para leitura e para desenhar a matriz de confusão; os milhares de arquivos intermediários da execução foram retirados do repositório.

A linha de cada classe na matriz é a classe real; a coluna é a previsão. O modelo de referência é a cascata com porta de atividade e agregação temporal. Acurácia global pode esconder erros importantes em classes menores; por isso também observamos precisão e sensibilidade de *Aedes* e falsos positivos por classe.

In [ ]:
REFERENCE = {
  "run_id": "ternary_verified_2026_10_14_corrections",
  "model": "activity_gated_log_odds_cascade_event",
  "evaluation_unit": "group",
  "total_groups": 1494,
  "groups_per_class": {
    "no_mosquito": 426,
    "other_mosquito": 1008,
    "aedes_aegypti": 60
  },
  "windows_retained": 42029,
  "training_windows_by_class": {
    "no_mosquito": 8658,
    "other_mosquito": 31205,
    "aedes_aegypti": 2166
  },
  "accuracy": 0.9002677376171352,
  "balanced_accuracy": 0.8367221849616215,
  "aedes_recall": 0.7833333333333333,
  "aedes_precision": 0.6714285714285714,
  "noise_to_aedes_fpr": 0.0,
  "other_to_aedes_fpr": 0.022817460317460316,
  "noise_to_any_mosquito_fpr": 0.23943661971830985,
  "counts": {
    "aedes_tp": 47,
    "aedes_total": 60,
    "noise_fp_to_aedes": 0,
    "noise_total": 426,
    "other_fp_to_aedes": 23,
    "other_total": 1008,
    "noise_fp_to_any_mosquito": 102
  },
  "confusion_matrix": [
    [
      324,
      102,
      0
    ],
    [
      11,
      974,
      23
    ],
    [
      0,
      13,
      47
    ]
  ],
  "classes": [
    "no_mosquito",
    "other_mosquito",
    "aedes_aegypti"
  ]
}

reference_table = pd.DataFrame([
    ("Grupos avaliados (unidade principal)", REFERENCE["total_groups"]),
    ("Janelas retidas para desenvolvimento", REFERENCE["windows_retained"]),
    ("Acurácia por grupo", f"{REFERENCE['accuracy']:.1%}"),
    ("Acurácia balanceada por grupo", f"{REFERENCE['balanced_accuracy']:.1%}"),
    ("Sensibilidade de Aedes", f"{REFERENCE['aedes_recall']:.1%} ({REFERENCE['counts']['aedes_tp']}/{REFERENCE['counts']['aedes_total']})"),
    ("Precisão de Aedes", f"{REFERENCE['aedes_precision']:.1%}"),
    ("Ruído classificado como Aedes", f"{REFERENCE['counts']['noise_fp_to_aedes']}/{REFERENCE['counts']['noise_total']}"),
    ("Outros mosquitos classificados como Aedes", f"{REFERENCE['counts']['other_fp_to_aedes']}/{REFERENCE['counts']['other_total']}"),
    ("Ruído classificado como algum mosquito", f"{REFERENCE['counts']['noise_fp_to_any_mosquito']}/{REFERENCE['counts']['noise_total']}"),
], columns=["Medida", "Resultado"])
display(reference_table)
print("Contagem de grupos por classe:", REFERENCE["groups_per_class"])

In [ ]:
labels = ["Sem mosquito", "Outro mosquito", "Aedes aegypti"]
display_cm = ConfusionMatrixDisplay(
    confusion_matrix=np.asarray(REFERENCE["confusion_matrix"]),
    display_labels=labels,
)
display_cm.plot(cmap="Blues", values_format="d", colorbar=False)
plt.title("Referência OOF por grupo — linhas reais, colunas previstas")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## 5. Como funciona o treino e o teste

A função `train()` prepara uma rodada imutável, audita identidades de áudio, separa grupos por blocos PCM e congela os modelos antes de qualquer previsão na dobra externa. A função `evaluate()` então prevê cada grupo externo uma única vez e calcula métricas por evento/gravação.

A política principal da referência detectou 47 dos 60 grupos *Aedes* (78,3%) e não emitiu *Aedes* para os 426 grupos de ruído. Ainda classificou 23 dos 1.008 grupos de outros mosquitos como *Aedes*. A meta amostral definida para essa confusão era no máximo 19; portanto, **o critério não foi atingido**. Além disso, 102/426 grupos de ruído receberam alguma classe de mosquito.

Esses denominadores pertencem a este corpus. Grupos não são necessariamente indivíduos biológicos independentes e os intervalos binomiais não transformam essa avaliação em validação de campo.

### Iniciar uma rodada nova

O treino completo está desativado para que abrir o caderno não dispare uma execução longa. Depois de preparar os três arquivos listados na seção 3, altere `RUN_NEW_TRAINING = True` na célula de configuração e execute as células em ordem. O treino usa cinco dobras externas e quatro internas por padrão. Cada rodada recebe um nome UTC próprio e não substitui outra.

In [ ]:
TRAINED_RUN = None
if RUN_NEW_TRAINING:
    from train_ternary_nested_models import train, evaluate

    run_id = datetime.now(timezone.utc).strftime("ternary_notebook_%Y%m%dT%H%M%SZ")
    TRAINED_RUN = ROOT / "results" / run_id
    train(TRAINED_RUN, threads=THREADS)
    evaluation = evaluate(TRAINED_RUN, threads=THREADS)
    print("Treino e teste concluídos em:", TRAINED_RUN)
else:
    print("Nenhum treino iniciado. A tabela e a matriz acima mostram a referência congelada.")

## 6. Ler os resultados da rodada nova

Ao habilitar o treino, esta célula carrega as métricas e previsões OOF recém-calculadas. A matriz mostra onde cada classe foi confundida. Examine principalmente falso positivo de *Aedes* em outros mosquitos e em ruído, além da cobertura: um sistema que se abstém muito pode apresentar boa precisão entre poucas emissões.

In [ ]:
if TRAINED_RUN is not None:
    metrics_path = TRAINED_RUN / "event_level_metrics.json"
    predictions_path = TRAINED_RUN / "event_predictions.csv"
    if not metrics_path.is_file() or not predictions_path.is_file():
        raise FileNotFoundError("A etapa de avaliação não gerou os artefatos esperados.")
    fresh = json.loads(metrics_path.read_text())
    model_key = "activity_gated_log_odds_cascade_event"
    scores = fresh["models"][model_key]
    counts = scores["integer_counts"]
    display(pd.DataFrame([
        ("Execução", fresh["run_id"]),
        ("Acurácia", f"{scores['accuracy']:.1%}"),
        ("Acurácia balanceada", f"{scores['balanced_accuracy']:.1%}"),
        ("Sensibilidade Aedes", f"{counts['aedes_tp']}/{counts['aedes_total']}"),
        ("Ruído → Aedes", f"{counts['noise_fp_to_aedes']}/{counts['noise_total']}"),
        ("Outros → Aedes", f"{counts['other_fp_to_aedes']}/{counts['other_total']}"),
    ], columns=["Medida", "Resultado"]))
    event_table = pd.read_csv(predictions_path)
    matrix = pd.crosstab(event_table.true_label, event_table[model_key + "_pred"]).reindex(
        index=[0, 1, 2], columns=[0, 1, 2], fill_value=0
    ).to_numpy()
    ConfusionMatrixDisplay(matrix, display_labels=labels).plot(cmap="Blues", values_format="d", colorbar=False)
    plt.title("Rodada nova: teste OOF por grupo")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("A matriz acima corresponde à referência; ainda não houve treino nesta sessão.")

## 7. Aplicar o modelo a uma gravação WAV

A inferência divide o sinal em janelas reais, calcula os mesmos descritores usados no treino, verifica a integridade da rodada congelada e agrega a decisão ao nível da gravação. Trechos curtos não são repetidos nem preenchidos artificialmente.

Para aplicar o modelo criado aqui: configure `AUDIO_PATH` na seção 2, habilite e execute o treino, conclua o teste e depois rode esta célula. A saída é uma decisão experimental; não é confirmação biológica.

In [ ]:
if AUDIO_PATH is None:
    print("Defina AUDIO_PATH com o caminho de um WAV para executar a inferência.")
elif TRAINED_RUN is None:
    print("Treine e teste uma rodada nesta sessão antes da aplicação; nenhum modelo binário foi publicado no repositório.")
else:
    from predict_ternary import predict_audio

    audio_path = Path(AUDIO_PATH).expanduser()
    if not audio_path.is_absolute():
        audio_path = ROOT / audio_path
    if not audio_path.is_file():
        raise FileNotFoundError(f"Gravação não encontrada: {audio_path}")
    application = predict_audio(TRAINED_RUN, audio_path)
    display(pd.Series(application, name="Resultado da aplicação").to_frame())

## 8. Interpretar e comunicar com responsabilidade

- `no_mosquito`: o modelo não encontrou evidência suficiente para as classes de mosquito avaliadas.
- `other_mosquito`: candidato a mosquito fora de *Aedes aegypti*; a classe agrega diferentes espécies.
- `aedes_aegypti`: candidato da classe, sujeito a falsos positivos observados no teste.
- `invalid_audio`: entrada inválida segundo as verificações de qualidade.

A avaliação é retrospectiva e limitada aos grupos e às fontes deste corpus. Os dados podem compartilhar indivíduos, locais ou condições de gravação. Uma gravação externa pequena também não substitui uma coleta independente. Antes de qualquer uso operacional, seriam necessárias novas gravações com indivíduos/sessões separados, rótulos biológicos confirmados, ruído sem mosquito e protocolo definido antes do teste.

## Arquivos de implementação usados pelo caderno

O notebook é o roteiro único de uso e ensino. Os módulos abaixo permanecem separados para que o congelamento por hash, as funções de treino e a inferência usem a mesma implementação auditável:

- `ternary_protocol.py`: leitura, filtragem, classes, grupos, auditoria PCM e regras temporais.
- `train_ternary_nested_models.py`: ajuste aninhado, modelos, calibração interna e congelamento.
- `evaluate_ternary_event_pipeline.py` e `ternary_reporting.py`: previsões externas e métricas por grupo.
- `predict_ternary.py`: aplicação em uma gravação nova.
- `aedes_classical.py` e `bioacoustic_invariance.py`: extração dos descritores acústicos.

O manifesto e o mapa de grupos ficam em `data/metadata/`. Áudios e cache de descritores são entradas locais e não são versionados neste repositório.